# Information Retrieval in the 1914 French Press

## Fetching the Data

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s french-press-1914

In [ ]:
!ls french-press-1914/le_figaro/1914/

## Corpus

This corpus is made of articles from the French press of 1914.

One folder per newspaper, each containing one JSON file per publication date.

The JSON files contain several fields, three of which we'll use in the rest of this lab:

- `contentAsText`, a list of strings (paragraphs are delimited in these strings by line breaks — `\n`)
- `date`, the newspaper's publication date, represented by a list containing one string (if the `content` variable holds the loaded JSON file, access it with `content["date"][0]`)
- `title`, the newspaper's name, represented by a list containing one string (if the `content` variable holds the loaded JSON file, access it with `content["title"][0]`)

Implement a function that takes the corpus path as input and returns a 3-column `numpy.ndarray`:

- the first column contains the paragraphs of the corpus editions (so one row per paragraph of the corpus), meeting the following conditions:
  - only paragraphs of more than 10 words
  - lowercase
  - without punctuation
- the second column contains the newspaper the paragraph comes from
- the third column contains the date of the edition the paragraph comes from

In [ ]:
# Your code here

### Solution

In [ ]:
import datetime
import json
import pathlib
import re

import numpy
import tqdm.notebook


def preprocess_text(text: str) -> str:
  out = re.sub(r"\W+", " ", text)
  return out.lower()


def get_data(directory: pathlib.Path) -> numpy.ndarray:
  data = []
  for edition in tqdm.notebook.tqdm(list(directory.rglob("*.json"))):
    with open(edition, encoding="utf8") as fh:
      content = json.load(fh)
    newspaper = content["title"][0]
    date = datetime.date.fromisoformat(content["date"][0]).strftime("%d/%m/%Y")
    for item in content["contentAsText"]:
      for paragraph in item.split("\n"):
        processed_paragraph = preprocess_text(paragraph)
        if len(processed_paragraph.split()) > 10:
          data.append((processed_paragraph, newspaper, date))
  return numpy.array(data)


data = get_data(pathlib.Path("french-press-1914"))

## TF-IDF Representation

Use a [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) to build the matrix `X` of the corpus paragraphs turned into TF-IDF weight vectors.

In [ ]:
# Your code here

### Solution

In [ ]:
import sklearn.feature_extraction.text

vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(data[:, 0])

## Similarity

Implement a function `best_score(query: str) -> None` that:

- takes a query string as input
- displays the newspapers and dates of the 5 best results in terms of cosine similarity

In [ ]:
# Your code here

### Solution

In [ ]:
import sklearn.metrics.pairwise


def best_score(query: str) -> None:
  query_vector = vectorizer.transform([query.lower()])
  cosine = sklearn.metrics.pairwise.cosine_similarity(X, query_vector)[:, 0]
  # We use the opposite of the cosine array in the argsort call to get the
  # indices of the array sorted by decreasing values, before keeping the
  # first 5
  top5 = (-cosine).argsort()[:5]
  for index, (text, newspaper, date), similarity in zip(top5,
                                                        data[top5],
                                                        cosine[top5]):
    print(f"{newspaper:10} on {date}; "
          f"similarity {similarity:5.2f}; "
          f"index {index:6}; "
          f"{text}")

## Search

Find the most relevant passages about:

- the assassination of Franz Ferdinand
- war against the Germans

Check your results by looking at the newspapers' text in `X`

In [ ]:
# Your code here

### Solution

In [ ]:
best_score("attentat François-Ferdinand")

In [ ]:
best_score("Guerre allemands")